# 06 — UK-AIR SOS Discovery
Downloads GetCapabilities and extracts observedProperty strings (best-effort).

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import requests

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def mask(s: str, keep=4):
    if not s: return ""
    s = str(s)
    return s[:keep] + "…" + "*"*6 if len(s) > keep else "*"*len(s)

def env(name: str, default=""):
    v = os.getenv(name, default)
    return v.strip() if isinstance(v, str) else v

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": [],
    }

def add_artifact(manifest: dict, path: Path, kind: str, meta=None):
    meta = meta or {}
    manifest["artifacts"].append({
        "kind": kind,
        "path": str(path.relative_to(PROJECT_ROOT)),
        "sha256": sha256_file(path) if path.exists() and path.is_file() else None,
        "meta": meta,
    })

print("PROJECT_ROOT:", PROJECT_ROOT)
print("UTC now:", utcnow())

In [ ]:
import xml.etree.ElementTree as ET
step = "06_ukair_catalogue_discovery"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

base = "https://uk-air.defra.gov.uk/sos-ukair/service"
r = requests.get(base, params={"service":"SOS","request":"GetCapabilities","acceptVersions":"2.0.0"}, timeout=60)

cap_path = out/"GetCapabilities.xml"
cap_path.write_bytes(r.content)
print("Capabilities HTTP:", r.status_code, "bytes:", len(r.content))

observed = []
try:
    root = ET.fromstring(r.content)
    for el in root.iter():
        tag = el.tag.split("}")[-1].lower()
        if tag in ("observedproperty","observableproperty"):
            if el.text and el.text.strip():
                observed.append(el.text.strip())
except Exception as e:
    print("Parse warning:", e)

observed = sorted(set(observed))
write_json(out/"observed_properties_from_capabilities.json", observed)

manifest = manifest_base(step, config_paths=[CONFIGS/"run.yml"])
add_artifact(manifest, cap_path, "raw_xml")
add_artifact(manifest, out/"observed_properties_from_capabilities.json", "observed_properties", meta={"count": len(observed)})
write_json(out/"manifest.json", manifest)

print("Observed properties found:", len(observed))
print("Wrote:", out/"manifest.json")